In [1]:
!pip install paddlepaddle-gpu==3.3.0 -i https://www.paddlepaddle.org.cn/packages/stable/cu129/

Looking in indexes: https://www.paddlepaddle.org.cn/packages/stable/cu129/
     ━━━━━━━━━━━╸━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.7/2.3 GB 16.8 MB/s eta 0:01:38
ERROR: Exception:
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pip/_vendor/urllib3/response.py", line 438, in _error_catcher
    yield
  File "/usr/local/lib/python3.12/dist-packages/pip/_vendor/urllib3/response.py", line 561, in read
    data = self._fp_read(amt) if not fp_closed else b""
           ^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_vendor/urllib3/response.py", line 527, in _fp_read
    return self._fp.read(amt) if amt is not None else self._fp.read()
           ^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_vendor/cachecontrol/filewrapper.py", line 98, in read
    data: bytes = self.__fp.read(amt)
                  ^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/http/client.py", line 479, in read
    s = self.fp.read(amt)
        ^^^

In [2]:
!pip install paddleocr "paddlex[ocr]"

In [3]:
!pip install "langchain>=0.1,<1.0"

In [4]:
from google.colab import drive
drive.flush_and_unmount()

Drive not mounted, so nothing to flush and unmount.


In [5]:
from google.colab import drive
drive.mount('/content/drive')


ValueError: Mountpoint must not already contain files

In [ ]:


pdf_path = "/content/drive/MyDrive/pdf_folder/sperated_small.pdf"
input_file = pdf_path  


import os
print("OK" if os.path.exists(pdf_path) else "OK:", pdf_path)


OK: /content/drive/MyDrive/pdf_folder/sperated_small.pdf


In [ ]:

import json
from pathlib import Path
from paddleocr import PPStructureV3
output_dir = Path("/content/drive/MyDrive/pdf_folder/ocr_results")
output_dir.mkdir(parents=True, exist_ok=True)

input_file = Path("/content/drive/MyDrive/pdf_folder/main.pdf") 

pipeline = PPStructureV3(lang="de", device="gpu")
output = pipeline.predict(input=str(input_file))

markdown_list = []
footnote_data = []
markdown_images = [] 

for res in output:
    md_info = res.markdown
    markdown_list.append(md_info)
    markdown_images.append(md_info.get("markdown_images", {})) 
    
    blocks = res.json["res"]["parsing_res_list"]

    number = next(
        (b["block_content"] for b in blocks if b["block_label"] == "number"),
        None,
    )
    footnotes = [
        b["block_content"]
        for b in blocks
        if b["block_label"] == "footnote"
    ]
    if footnotes:
        footnote_data.append({"number": number, "footnote": footnotes})


markdown_result = pipeline.concatenate_markdown_pages(markdown_list)
markdown_texts = markdown_result.markdown["markdown_texts"]
stem = Path(input_file).stem
mkd_path = output_dir / f"{stem}.md"
with open(mkd_path, "w", encoding="utf-8") as f:
    f.write(markdown_texts)
print(f"{mkd_path}")


json_path = output_dir / f"{stem}_footnotes.json"
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(footnote_data, f, ensure_ascii=False, indent=4)
print(f"{json_path}")

for item in markdown_images:
    if item:
        for path, image in item.items():
            file_path = output_dir / path
            file_path.parent.mkdir(parents=True, exist_ok=True)
            image.save(file_path)

ValueError: Mountpoint must not already contain files